In [1]:
from datetime import date
import pandas as pd

from engine_product.pricing import YieldProblem, yield_to_maturity
from engine_product.risk import macaulay_duration, modified_duration
from engine_product.instruments import NTNFContract

from engine_product.calendars.repository import DataFrameCalendarRepository
from engine_product.calendars.business_calendar import BusinessCalendar
from engine_product.convention.conventions import BU252

import duckdb
from pathlib import Path

path_db = Path(r"C:\Users\Dell\OneDrive\Documentos\GitHub\ML-ETTJ26\data\duckdb\ml_ettj26.duckdb")
con = duckdb.connect(path_db)
con.sql("SHOW TABLES")

┌────────────────────────────────────────────────────────┐
│                          name                          │
│                        varchar                         │
├────────────────────────────────────────────────────────┤
│ vw_refined_b3_forwards_di1                             │
│ vw_refined_b3_swaps_di_pre                             │
│ vw_refined_bcb_demab_government_bonds_secondary_market │
│ vw_refined_bcb_demab_ltn                               │
│ vw_refined_bcb_demab_ntnf                              │
│ vw_refined_calendar_br                                 │
│ vw_refined_ipca                                        │
│ vw_refined_selic                                       │
│ vw_trusted_b3_forwards_di1_lineage                     │
│ vw_trusted_b3_forwards_di1_quotes                      │
│ vw_trusted_b3_forwards_metadata                        │
│ vw_trusted_b3_swaps_dixpre_quotes                      │
│ vw_trusted_b3_swaps_lineage                           

In [2]:
calendar_df = con.sql("SELECT * FROM vw_refined_calendar_br").df().reset_index(drop=True)
calendar_df["act_index"] = calendar_df.index
calendar_df["date"] = pd.to_datetime(calendar_df["date"]).dt.date

calendar_repo = DataFrameCalendarRepository(calendar_df)
calendar_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 36159 entries, 0 to 36158
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   calendar_id       36159 non-null  str   
 1   date              36159 non-null  object
 2   year              36159 non-null  int32 
 3   month             36159 non-null  int32 
 4   day               36159 non-null  int32 
 5   weekday           36159 non-null  int32 
 6   is_weekend        36159 non-null  bool  
 7   is_holiday        36159 non-null  bool  
 8   is_business_day   36159 non-null  bool  
 9   act_index         36159 non-null  int64 
 10  bd_index          36159 non-null  int64 
 11  holiday_name      1263 non-null   str   
 12  source_file_hash  36159 non-null  str   
dtypes: bool(3), int32(4), int64(2), object(1), str(3)
memory usage: 4.9+ MB


In [3]:
calendar = BusinessCalendar(calendar_repo)
bu252 = BU252(calendar)
bu252.year_fraction(date(2024, 6, 10), date(2024, 12, 10))

0.5119047619047619

In [4]:

as_of_date = date(2026, 1, 30)

ntnf = NTNFContract(
    start_date=as_of_date,
    maturity_date=date(2029, 1, 1),
    calendar=calendar,
    day_count=bu252,
)

cashflows = ntnf.build_cashflows(as_of_date=as_of_date)

problem = YieldProblem(
    cashflows=cashflows,
    market_price=946.2175,
    settlement_date=as_of_date,
    day_count=bu252,
)

result = yield_to_maturity(problem)

mac = macaulay_duration(
    cashflows=cashflows,
    ytm=result.ytm,
    settlement_date=as_of_date,
    day_count=bu252,
)

mod = modified_duration(
    cashflows=cashflows,
    ytm=result.ytm,
    settlement_date=as_of_date,
    day_count=bu252,
)

In [5]:
result

YieldSolverResult(ytm=0.12857751523282882, method=<YieldSolverMethod.NEWTON: 'NEWTON'>, iterations=4)

In [6]:
cashflows

[Cashflow(payment_date=datetime.date(2026, 7, 1), amount=48.808848170151634, cashflow_type=<CashflowType.INTEREST: 'INTEREST'>, accrual_start=datetime.date(2026, 1, 30), accrual_end=datetime.date(2026, 7, 1), notional_before=1000.0, notional_after=1000.0, metadata={'accrual_factor': 0.04880884817015163}),
 Cashflow(payment_date=datetime.date(2027, 1, 4), amount=48.808848170151634, cashflow_type=<CashflowType.INTEREST: 'INTEREST'>, accrual_start=datetime.date(2026, 7, 1), accrual_end=datetime.date(2027, 1, 4), notional_before=1000.0, notional_after=1000.0, metadata={'accrual_factor': 0.04880884817015163}),
 Cashflow(payment_date=datetime.date(2027, 7, 1), amount=48.808848170151634, cashflow_type=<CashflowType.INTEREST: 'INTEREST'>, accrual_start=datetime.date(2027, 1, 4), accrual_end=datetime.date(2027, 7, 1), notional_before=1000.0, notional_after=1000.0, metadata={'accrual_factor': 0.04880884817015163}),
 Cashflow(payment_date=datetime.date(2028, 1, 3), amount=48.808848170151634, cash